# MODULE 1.2: Connection Configuration

This notebook sets up secure connection configuration for the Personal Expense Tracker Lakebase PostgreSQL instance.

## Objectives
- Configure secure connection string with SSL
- Set up authentication with proper credential management
- Implement comprehensive error handling
- Test database connections
- Ensure proper resource cleanup
- Use environment variables for sensitive data
- Support connection pooling for production use
- Follow security best practices


In [ ]:
# Install required packages
%pip install databricks-sdk --quiet
%pip install python-dotenv --quiet
%pip install psycopg2-binary --quiet
%pip install sqlalchemy --quiet


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Optional, Dict, Any
from contextlib import contextmanager
from dotenv import load_dotenv
import psycopg2
from psycopg2 import pool, sql
from psycopg2.extras import RealDictCursor
from psycopg2.pool import ThreadedConnectionPool
from sqlalchemy import create_engine, event
from sqlalchemy.pool import QueuePool
from databricks.sdk import WorkspaceClient

# Load environment variables
load_dotenv()

# Add utils directory to path
PROJECT_ROOT = Path("/Workspace/Repos/Personal Expense Tracker")
sys.path.insert(0, str(PROJECT_ROOT))

print("✓ Required packages imported successfully")


## Load Configuration

Load the Lakebase configuration created in MODULE 1.1 and set up paths.


In [ ]:
# Configuration paths
CONFIG_DIR = PROJECT_ROOT / "configuration"
LAKEBASE_CONFIG_FILE = CONFIG_DIR / "lakebase-config.json"
UTILS_DIR = PROJECT_ROOT / "utils"
UTILS_DIR.mkdir(exist_ok=True)

def load_lakebase_config() -> Dict[str, Any]:
    """Load Lakebase configuration from JSON file."""
    try:
        if not LAKEBASE_CONFIG_FILE.exists():
            raise FileNotFoundError(
                f"Configuration file not found: {LAKEBASE_CONFIG_FILE}\n"
                "Please run MODULE 1.1 (01-setup-lakebase.ipynb) first."
            )
        
        with open(LAKEBASE_CONFIG_FILE, 'r') as f:
            config = json.load(f)
        
        print("✓ Configuration loaded successfully")
        return config
    except Exception as e:
        print(f"❌ Error loading configuration: {str(e)}")
        raise

# Load configuration
config = load_lakebase_config()
print(f"Instance Name: {config.get('instance', {}).get('name', 'N/A')}")
print(f"Instance Status: {config.get('instance', {}).get('status', 'N/A')}")


## Environment Variables and Secrets Management

Set up secure credential management using environment variables and Databricks Secrets.


In [ ]:
def get_credentials_from_secrets() -> Optional[Dict[str, str]]:
    """
    Retrieve database credentials from Databricks Secrets.
    This is the preferred method for production environments.
    """
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        
        # Try to get credentials from Databricks Secrets
        # Format: secrets.get(scope, key)
        # Example: secrets.get("expense-tracker", "db_username")
        
        # For now, we'll use a placeholder - replace with actual secret scope
        SECRET_SCOPE = os.getenv("DB_SECRET_SCOPE", "expense-tracker")
        
        try:
            # Note: This requires dbutils in Databricks runtime
            # In notebook environment, use: dbutils.secrets.get(scope, key)
            username = os.getenv("DB_USERNAME") or config.get('connection', {}).get('credentials', {}).get('username')
            password = os.getenv("DB_PASSWORD") or config.get('connection', {}).get('credentials', {}).get('password')
            
            if username and password:
                return {"username": username, "password": password}
        except Exception as e:
            print(f"⚠️  Could not retrieve from secrets: {str(e)}")
        
        return None
    except Exception as e:
        print(f"⚠️  Secrets not available: {str(e)}")
        return None

def get_credentials() -> Dict[str, str]:
    """
    Get database credentials with fallback priority:
    1. Environment variables (most secure)
    2. Databricks Secrets
    3. Configuration file (development only)
    """
    # Priority 1: Environment variables
    username = os.getenv("DB_USERNAME")
    password = os.getenv("DB_PASSWORD")
    
    if username and password:
        print("✓ Using credentials from environment variables")
        return {"username": username, "password": password}
    
    # Priority 2: Databricks Secrets
    secrets_creds = get_credentials_from_secrets()
    if secrets_creds:
        print("✓ Using credentials from Databricks Secrets")
        return secrets_creds
    
    # Priority 3: Configuration file (development only)
    conn_config = config.get('connection', {})
    credentials = conn_config.get('credentials', {})
    
    if credentials.get('username') and credentials.get('password'):
        print("⚠️  Using credentials from configuration file (development only)")
        print("   For production, use environment variables or Databricks Secrets")
        return credentials
    
    raise ValueError(
        "No database credentials found. Set DB_USERNAME and DB_PASSWORD "
        "environment variables or configure Databricks Secrets."
    )

# Get credentials
credentials = get_credentials()
print(f"Username: {credentials['username']}")
print(f"Password: {'*' * len(credentials['password'])}")


In [ ]:
def build_connection_string(
    host: str,
    port: int,
    database: str,
    username: str,
    password: str,
    ssl_mode: str = "require",
    ssl_cert: Optional[str] = None,
    ssl_key: Optional[str] = None,
    ssl_root_cert: Optional[str] = None
) -> str:
    """
    Build a secure PostgreSQL connection string with SSL support.
    
    Args:
        host: Database host
        port: Database port
        database: Database name
        username: Database username
        password: Database password
        ssl_mode: SSL mode (disable, allow, prefer, require, verify-ca, verify-full)
        ssl_cert: Path to SSL certificate (optional)
        ssl_key: Path to SSL key (optional)
        ssl_root_cert: Path to root certificate (optional)
    
    Returns:
        PostgreSQL connection string
    """
    # Base connection string
    conn_str = f"postgresql://{username}:{password}@{host}:{port}/{database}"
    
    # SSL parameters
    ssl_params = [f"sslmode={ssl_mode}"]
    
    if ssl_cert:
        ssl_params.append(f"sslcert={ssl_cert}")
    if ssl_key:
        ssl_params.append(f"sslkey={ssl_key}")
    if ssl_root_cert:
        ssl_params.append(f"sslrootcert={ssl_root_cert}")
    
    # Append SSL parameters
    if ssl_params:
        conn_str += "?" + "&".join(ssl_params)
    
    return conn_str

def get_connection_string() -> str:
    """Get connection string from configuration with credentials."""
    conn_config = config.get('connection', {})
    
    host = conn_config.get('host')
    port = conn_config.get('port', 5432)
    database = conn_config.get('database', 'postgres')
    ssl_mode = conn_config.get('ssl_mode', 'require')
    
    if not host:
        raise ValueError("Database host not found in configuration")
    
    return build_connection_string(
        host=host,
        port=port,
        database=database,
        username=credentials['username'],
        password=credentials['password'],
        ssl_mode=ssl_mode
    )

# Build connection string
connection_string = get_connection_string()

# Display connection info (masked)
conn_info = connection_string.split('@')[1] if '@' in connection_string else "N/A"
print(f"✓ Connection string built successfully")
print(f"Host: {conn_info.split('/')[0] if '/' in conn_info else 'N/A'}")
print(f"SSL Mode: {config.get('connection', {}).get('ssl_mode', 'require')}")


In [ ]:
class DatabaseConnection:
    """
    Secure database connection manager with error handling and resource cleanup.
    """
    
    def __init__(self, connection_string: str, pool_size: int = 5, max_overflow: int = 10):
        """
        Initialize database connection manager.
        
        Args:
            connection_string: PostgreSQL connection string
            pool_size: Number of connections to maintain in pool
            max_overflow: Maximum overflow connections
        """
        self.connection_string = connection_string
        self.pool_size = pool_size
        self.max_overflow = max_overflow
        self._pool: Optional[ThreadedConnectionPool] = None
        self._engine = None
    
    def create_pool(self) -> ThreadedConnectionPool:
        """Create a connection pool."""
        if self._pool is None:
            try:
                # Parse connection string for pool creation
                from urllib.parse import urlparse
                parsed = urlparse(self.connection_string)
                
                self._pool = ThreadedConnectionPool(
                    minconn=1,
                    maxconn=self.pool_size,
                    host=parsed.hostname,
                    port=parsed.port or 5432,
                    database=parsed.path[1:] if parsed.path else 'postgres',
                    user=parsed.username,
                    password=parsed.password,
                    sslmode=parsed.query.split('sslmode=')[1].split('&')[0] if 'sslmode=' in parsed.query else 'require'
                )
                print(f"✓ Connection pool created (size: {self.pool_size})")
            except Exception as e:
                print(f"❌ Error creating connection pool: {str(e)}")
                raise
        return self._pool
    
    def get_connection(self):
        """Get a connection from the pool."""
        if self._pool is None:
            self.create_pool()
        
        try:
            return self._pool.getconn()
        except Exception as e:
            raise ConnectionError(f"Failed to get connection from pool: {str(e)}")
    
    def return_connection(self, conn):
        """Return a connection to the pool."""
        if self._pool:
            try:
                self._pool.putconn(conn)
            except Exception as e:
                print(f"⚠️  Error returning connection to pool: {str(e)}")
    
    @contextmanager
    def get_cursor(self, commit: bool = False):
        """
        Context manager for database cursor with automatic cleanup.
        
        Args:
            commit: Whether to commit transaction on exit
        
        Yields:
            Database cursor
        """
        conn = None
        cursor = None
        try:
            conn = self.get_connection()
            cursor = conn.cursor(cursor_factory=RealDictCursor)
            yield cursor
            if commit:
                conn.commit()
        except Exception as e:
            if conn:
                conn.rollback()
            raise ConnectionError(f"Database operation failed: {str(e)}")
        finally:
            if cursor:
                cursor.close()
            if conn:
                self.return_connection(conn)
    
    def get_sqlalchemy_engine(self):
        """Get SQLAlchemy engine with connection pooling."""
        if self._engine is None:
            try:
                self._engine = create_engine(
                    self.connection_string,
                    poolclass=QueuePool,
                    pool_size=self.pool_size,
                    max_overflow=self.max_overflow,
                    pool_pre_ping=True,  # Verify connections before using
                    echo=False  # Set to True for SQL logging
                )
                print(f"✓ SQLAlchemy engine created with connection pooling")
            except Exception as e:
                print(f"❌ Error creating SQLAlchemy engine: {str(e)}")
                raise
        return self._engine
    
    def close_all(self):
        """Close all connections and cleanup resources."""
        if self._pool:
            try:
                self._pool.closeall()
                print("✓ Connection pool closed")
            except Exception as e:
                print(f"⚠️  Error closing pool: {str(e)}")
        
        if self._engine:
            try:
                self._engine.dispose()
                print("✓ SQLAlchemy engine disposed")
            except Exception as e:
                print(f"⚠️  Error disposing engine: {str(e)}")

# Initialize database connection manager
db_connection = DatabaseConnection(connection_string, pool_size=5, max_overflow=10)
print("✓ Database connection manager initialized")


In [ ]:
def test_connection(connection_string: str) -> Dict[str, Any]:
    """
    Test database connection with comprehensive error handling.
    
    Returns:
        Dictionary with test results
    """
    results = {
        "success": False,
        "postgresql_version": None,
        "database_name": None,
        "current_user": None,
        "ssl_enabled": None,
        "connection_time_ms": None,
        "error": None
    }
    
    import time
    start_time = time.time()
    
    try:
        # Test basic connection
        conn = psycopg2.connect(connection_string)
        
        with conn.cursor() as cursor:
            # Test 1: PostgreSQL version
            cursor.execute("SELECT version();")
            results["postgresql_version"] = cursor.fetchone()[0]
            
            # Test 2: Current database
            cursor.execute("SELECT current_database();")
            results["database_name"] = cursor.fetchone()[0]
            
            # Test 3: Current user
            cursor.execute("SELECT current_user;")
            results["current_user"] = cursor.fetchone()[0]
            
            # Test 4: SSL status
            cursor.execute("SHOW ssl;")
            ssl_status = cursor.fetchone()[0]
            results["ssl_enabled"] = ssl_status == "on"
        
        conn.close()
        results["success"] = True
        results["connection_time_ms"] = round((time.time() - start_time) * 1000, 2)
        
    except psycopg2.OperationalError as e:
        results["error"] = f"Operational Error: {str(e)}"
        results["success"] = False
    except psycopg2.ProgrammingError as e:
        results["error"] = f"Programming Error: {str(e)}"
        results["success"] = False
    except Exception as e:
        results["error"] = f"Unexpected Error: {str(e)}"
        results["success"] = False
    
    return results

# Test connection
print("\\n🧪 Testing database connection...")
test_results = test_connection(connection_string)

print("\\n" + "="*60)
print("CONNECTION TEST RESULTS")
print("="*60)
if test_results["success"]:
    print("✅ Connection successful!")
    print(f"PostgreSQL Version: {test_results['postgresql_version']}")
    print(f"Database: {test_results['database_name']}")
    print(f"User: {test_results['current_user']}")
    print(f"SSL Enabled: {test_results['ssl_enabled']}")
    print(f"Connection Time: {test_results['connection_time_ms']} ms")
else:
    print("❌ Connection failed!")
    print(f"Error: {test_results['error']}")
print("="*60)


## Test Connection Pooling

Verify that connection pooling works correctly.


In [ ]:
def test_connection_pool(db_conn: DatabaseConnection, num_queries: int = 5) -> Dict[str, Any]:
    """
    Test connection pooling with multiple concurrent queries.
    
    Args:
        db_conn: Database connection manager
        num_queries: Number of queries to execute
    
    Returns:
        Dictionary with pool test results
    """
    results = {
        "success": False,
        "queries_executed": 0,
        "queries_failed": 0,
        "total_time_ms": None,
        "errors": []
    }
    
    import time
    start_time = time.time()
    
    try:
        # Create pool
        db_conn.create_pool()
        
        # Execute multiple queries using the pool
        for i in range(num_queries):
            try:
                with db_conn.get_cursor() as cursor:
                    cursor.execute("SELECT current_timestamp, current_database();")
                    result = cursor.fetchone()
                    results["queries_executed"] += 1
            except Exception as e:
                results["queries_failed"] += 1
                results["errors"].append(f"Query {i+1}: {str(e)}")
        
        results["success"] = results["queries_failed"] == 0
        results["total_time_ms"] = round((time.time() - start_time) * 1000, 2)
        
    except Exception as e:
        results["errors"].append(f"Pool test error: {str(e)}")
    
    return results

# Test connection pool
if test_results["success"]:
    print("\\n🧪 Testing connection pooling...")
    pool_test_results = test_connection_pool(db_connection, num_queries=5)
    
    print("\\n" + "="*60)
    print("CONNECTION POOL TEST RESULTS")
    print("="*60)
    if pool_test_results["success"]:
        print("✅ Connection pool working correctly!")
        print(f"Queries Executed: {pool_test_results['queries_executed']}")
        print(f"Total Time: {pool_test_results['total_time_ms']} ms")
    else:
        print("⚠️  Some queries failed")
        print(f"Successful: {pool_test_results['queries_executed']}")
        print(f"Failed: {pool_test_results['queries_failed']}")
        for error in pool_test_results['errors']:
            print(f"  - {error}")
    print("="*60)
else:
    print("\\n⏭️  Skipping pool test (connection failed)")


## Save Connection Utility Module

Save the connection utility as a reusable Python module for use across the project.


In [ ]:
# Save connection utility module
UTILS_DIR = PROJECT_ROOT / "utils"
UTILS_DIR.mkdir(exist_ok=True)

connection_utility_code = '''"""
Database Connection Utility Module
Provides secure database connection management with pooling and error handling.
"""

import os
import json
from pathlib import Path
from typing import Optional, Dict, Any
from contextlib import contextmanager
from dotenv import load_dotenv
import psycopg2
from psycopg2 import pool
from psycopg2.extras import RealDictCursor
from psycopg2.pool import ThreadedConnectionPool
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

# Load environment variables
load_dotenv()


class DatabaseConnection:
    """
    Secure database connection manager with error handling and resource cleanup.
    Supports connection pooling for production use.
    """
    
    def __init__(self, connection_string: str, pool_size: int = 5, max_overflow: int = 10):
        """
        Initialize database connection manager.
        
        Args:
            connection_string: PostgreSQL connection string
            pool_size: Number of connections to maintain in pool
            max_overflow: Maximum overflow connections
        """
        self.connection_string = connection_string
        self.pool_size = pool_size
        self.max_overflow = max_overflow
        self._pool: Optional[ThreadedConnectionPool] = None
        self._engine = None
    
    def create_pool(self) -> ThreadedConnectionPool:
        """Create a connection pool."""
        if self._pool is None:
            try:
                from urllib.parse import urlparse
                parsed = urlparse(self.connection_string)
                
                self._pool = ThreadedConnectionPool(
                    minconn=1,
                    maxconn=self.pool_size,
                    host=parsed.hostname,
                    port=parsed.port or 5432,
                    database=parsed.path[1:] if parsed.path else 'postgres',
                    user=parsed.username,
                    password=parsed.password,
                    sslmode=parsed.query.split('sslmode=')[1].split('&')[0] if 'sslmode=' in parsed.query else 'require'
                )
            except Exception as e:
                raise ConnectionError(f"Failed to create connection pool: {str(e)}")
        return self._pool
    
    def get_connection(self):
        """Get a connection from the pool."""
        if self._pool is None:
            self.create_pool()
        
        try:
            return self._pool.getconn()
        except Exception as e:
            raise ConnectionError(f"Failed to get connection from pool: {str(e)}")
    
    def return_connection(self, conn):
        """Return a connection to the pool."""
        if self._pool:
            try:
                self._pool.putconn(conn)
            except Exception as e:
                print(f"Warning: Error returning connection to pool: {str(e)}")
    
    @contextmanager
    def get_cursor(self, commit: bool = False):
        """
        Context manager for database cursor with automatic cleanup.
        
        Args:
            commit: Whether to commit transaction on exit
        
        Yields:
            Database cursor
        """
        conn = None
        cursor = None
        try:
            conn = self.get_connection()
            cursor = conn.cursor(cursor_factory=RealDictCursor)
            yield cursor
            if commit:
                conn.commit()
        except Exception as e:
            if conn:
                conn.rollback()
            raise ConnectionError(f"Database operation failed: {str(e)}")
        finally:
            if cursor:
                cursor.close()
            if conn:
                self.return_connection(conn)
    
    def get_sqlalchemy_engine(self):
        """Get SQLAlchemy engine with connection pooling."""
        if self._engine is None:
            try:
                self._engine = create_engine(
                    self.connection_string,
                    poolclass=QueuePool,
                    pool_size=self.pool_size,
                    max_overflow=self.max_overflow,
                    pool_pre_ping=True,
                    echo=False
                )
            except Exception as e:
                raise ConnectionError(f"Failed to create SQLAlchemy engine: {str(e)}")
        return self._engine
    
    def close_all(self):
        """Close all connections and cleanup resources."""
        if self._pool:
            try:
                self._pool.closeall()
            except Exception as e:
                print(f"Warning: Error closing pool: {str(e)}")
        
        if self._engine:
            try:
                self._engine.dispose()
            except Exception as e:
                print(f"Warning: Error disposing engine: {str(e)}")


def load_lakebase_config(config_path: Optional[Path] = None) -> Dict[str, Any]:
    """
    Load Lakebase configuration from JSON file.
    
    Args:
        config_path: Path to configuration file
    
    Returns:
        Configuration dictionary
    """
    if config_path is None:
        config_path = Path(__file__).parent.parent / "configuration" / "lakebase-config.json"
    
    if not config_path.exists():
        raise FileNotFoundError(f"Configuration file not found: {config_path}")
    
    with open(config_path, 'r') as f:
        return json.load(f)


def get_credentials(config: Optional[Dict[str, Any]] = None) -> Dict[str, str]:
    """
    Get database credentials with fallback priority:
    1. Environment variables (most secure)
    2. Configuration file (development only)
    
    Args:
        config: Optional configuration dictionary
    
    Returns:
        Dictionary with username and password
    """
    if config is None:
        config = load_lakebase_config()
    
    # Priority 1: Environment variables
    username = os.getenv("DB_USERNAME")
    password = os.getenv("DB_PASSWORD")
    
    if username and password:
        return {"username": username, "password": password}
    
    # Priority 2: Configuration file
    conn_config = config.get('connection', {})
    credentials = conn_config.get('credentials', {})
    
    if credentials.get('username') and credentials.get('password'):
        return credentials
    
    raise ValueError(
        "No database credentials found. Set DB_USERNAME and DB_PASSWORD "
        "environment variables."
    )


def build_connection_string(
    host: str,
    port: int,
    database: str,
    username: str,
    password: str,
    ssl_mode: str = "require"
) -> str:
    """
    Build a secure PostgreSQL connection string with SSL support.
    
    Args:
        host: Database host
        port: Database port
        database: Database name
        username: Database username
        password: Database password
        ssl_mode: SSL mode (require, verify-ca, verify-full)
    
    Returns:
        PostgreSQL connection string
    """
    return f"postgresql://{username}:{password}@{host}:{port}/{database}?sslmode={ssl_mode}"


def get_connection_string(config: Optional[Dict[str, Any]] = None) -> str:
    """
    Get connection string from configuration.
    
    Args:
        config: Optional configuration dictionary
    
    Returns:
        PostgreSQL connection string
    """
    if config is None:
        config = load_lakebase_config()
    
    conn_config = config.get('connection', {})
    credentials = get_credentials(config)
    
    host = conn_config.get('host')
    port = conn_config.get('port', 5432)
    database = conn_config.get('database', 'postgres')
    ssl_mode = conn_config.get('ssl_mode', 'require')
    
    if not host:
        raise ValueError("Database host not found in configuration")
    
    return build_connection_string(
        host=host,
        port=port,
        database=database,
        username=credentials['username'],
        password=credentials['password'],
        ssl_mode=ssl_mode
    )


def test_connection(connection_string: str) -> Dict[str, Any]:
    """
    Test database connection.
    
    Args:
        connection_string: PostgreSQL connection string
    
    Returns:
        Dictionary with test results
    """
    import time
    import psycopg2
    
    results = {
        "success": False,
        "postgresql_version": None,
        "database_name": None,
        "current_user": None,
        "ssl_enabled": None,
        "connection_time_ms": None,
        "error": None
    }
    
    start_time = time.time()
    
    try:
        conn = psycopg2.connect(connection_string)
        
        with conn.cursor() as cursor:
            cursor.execute("SELECT version();")
            results["postgresql_version"] = cursor.fetchone()[0]
            
            cursor.execute("SELECT current_database();")
            results["database_name"] = cursor.fetchone()[0]
            
            cursor.execute("SELECT current_user;")
            results["current_user"] = cursor.fetchone()[0]
            
            cursor.execute("SHOW ssl;")
            ssl_status = cursor.fetchone()[0]
            results["ssl_enabled"] = ssl_status == "on"
        
        conn.close()
        results["success"] = True
        results["connection_time_ms"] = round((time.time() - start_time) * 1000, 2)
        
    except Exception as e:
        results["error"] = str(e)
        results["success"] = False
    
    return results
'''

# Write the utility module
utility_file = UTILS_DIR / "db_connection.py"
with open(utility_file, 'w') as f:
    f.write(connection_utility_code)

print(f"✓ Connection utility module saved to: {utility_file}")


In [ ]:
def save_connection_config(
    connection_string: str,
    test_results: Dict[str, Any],
    config_path: Path
):
    """
    Save connection configuration with test results.
    
    Args:
        connection_string: Connection string (credentials masked)
        test_results: Connection test results
        config_path: Path to save configuration
    """
    # Mask password in connection string for storage
    masked_conn_str = connection_string
    if '@' in connection_string and ':' in connection_string:
        parts = connection_string.split('@')
        if ':' in parts[0]:
            user_pass = parts[0].split('://')[1] if '://' in parts[0] else parts[0]
            if ':' in user_pass:
                username = user_pass.split(':')[0]
                masked_conn_str = connection_string.replace(
                    user_pass.split(':')[1],
                    '***MASKED***'
                )
    
    connection_config = {
        "connection_string_masked": masked_conn_str,
        "connection_test": {
            "success": test_results["success"],
            "postgresql_version": test_results.get("postgresql_version"),
            "database_name": test_results.get("database_name"),
            "ssl_enabled": test_results.get("ssl_enabled"),
            "connection_time_ms": test_results.get("connection_time_ms")
        },
        "pool_config": {
            "pool_size": db_connection.pool_size,
            "max_overflow": db_connection.max_overflow
        },
        "security": {
            "ssl_mode": config.get('connection', {}).get('ssl_mode', 'require'),
            "credentials_source": "environment_variables" if os.getenv("DB_USERNAME") else "config_file"
        }
    }
    
    # Update existing config
    updated_config = config.copy()
    updated_config["connection_config"] = connection_config
    
    with open(config_path, 'w') as f:
        json.dump(updated_config, f, indent=2)
    
    print(f"✓ Connection configuration saved to: {config_path}")

# Save connection configuration
if test_results["success"]:
    save_connection_config(connection_string, test_results, LAKEBASE_CONFIG_FILE)
    
    # Display saved config (without sensitive data)
    print("\\n📄 Connection Configuration Summary:")
    print(json.dumps({
        "connection_test": test_results,
        "pool_config": {
            "pool_size": db_connection.pool_size,
            "max_overflow": db_connection.max_overflow
        },
        "security": {
            "ssl_enabled": test_results.get("ssl_enabled"),
            "ssl_mode": config.get('connection', {}).get('ssl_mode', 'require')
        }
    }, indent=2))
else:
    print("\\n⚠️  Connection configuration not saved (connection test failed)")


## Resource Cleanup

Ensure proper cleanup of database connections and pools.


In [ ]:
# Cleanup resources
print("\\n🧹 Cleaning up resources...")
db_connection.close_all()
print("✓ Resources cleaned up successfully")


## Usage Example

Example of how to use the connection utility in other notebooks or applications.


In [ ]:
# Example usage code (commented out - for reference)
"""
# Import the utility module
from utils.db_connection import (
    DatabaseConnection,
    get_connection_string,
    load_lakebase_config,
    test_connection
)

# Load configuration
config = load_lakebase_config()

# Get connection string
conn_str = get_connection_string(config)

# Create connection manager
db = DatabaseConnection(conn_str, pool_size=5, max_overflow=10)

# Test connection
test_results = test_connection(conn_str)
if test_results["success"]:
    print("Connection successful!")
    
    # Use connection with automatic cleanup
    with db.get_cursor(commit=True) as cursor:
        cursor.execute("SELECT current_database();")
        result = cursor.fetchone()
        print(f"Current database: {result['current_database']}")
    
    # Cleanup when done
    db.close_all()
else:
    print(f"Connection failed: {test_results['error']}")
"""

print("✓ Usage example documented")
print("\\nTo use in other notebooks:")
print("  from utils.db_connection import DatabaseConnection, get_connection_string")
print("  db = DatabaseConnection(get_connection_string())")


## Summary

✅ **Connection Configuration Complete**

The secure database connection is now configured and ready for use.

### What Was Configured:
- ✅ Secure connection string with SSL encryption
- ✅ Authentication with environment variable support
- ✅ Comprehensive error handling
- ✅ Connection testing and validation
- ✅ Connection pooling for production use
- ✅ Proper resource cleanup
- ✅ Reusable utility module (`utils/db_connection.py`)

### Security Features:
- SSL encryption enabled (require mode)
- Credentials from environment variables (preferred)
- Connection string masking in saved config
- Secure credential management

### Next Steps:
1. Set environment variables for production:
   ```bash
   export DB_USERNAME=your_username
   export DB_PASSWORD=your_password
   ```
2. Use the connection utility in other notebooks:
   ```python
   from utils.db_connection import DatabaseConnection, get_connection_string
   db = DatabaseConnection(get_connection_string())
   ```
3. Proceed to `03-create-schema.ipynb` to create database tables

### Important Notes:
- For production, always use environment variables or Databricks Secrets
- Connection pooling is enabled for better performance
- Always use context managers (`with db.get_cursor()`) for automatic cleanup
- Test connections before deploying to production
